In [12]:
import pandas as pd

df = pd.read_csv("src\data\df_final.csv")



<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\diego\AppData\Local\Temp\ipykernel_26640\2969422670.py:3: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("src\data\df_final.csv")


In [13]:
df.info()
df.head()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4562 entries, 0 to 4561
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        4562 non-null   float64
 1   estratopri  4540 non-null   object 
 2   aoj21       2908 non-null   object 
 3   colseg      2995 non-null   object 
 4   ur          1568 non-null   object 
 5   q1          1569 non-null   object 
 6   q2          4549 non-null   float64
 7   m1          2980 non-null   object 
 8   trust       1549 non-null   float64
 9   pr5         1476 non-null   float64
 10  ideologia   2347 non-null   object 
dtypes: float64(4), object(7)
memory usage: 392.2+ KB


,year,estratopri,aoj21,colseg,ur,q1,q2,m1,trust,pr5,ideologia
count,4562.000000,4540,2908,2995,1568,1569,4549.000000,2980,1549.000000,1476.000000,2347
unique,NaN,6,7,3,2,2,NaN,5,NaN,NaN,5
top,NaN,Oriental,Delincuentes comunes,Peor,Urbano,Mujer,NaN,"Ni bueno, ni malo (regular)",NaN,NaN,Centro
freq,NaN,1134,1298,2101,1428,796,NaN,947,NaN,NaN,658
mean,2022.375712,NaN,NaN,NaN,NaN,NaN,40.792922,NaN,4.063267,3.939024,NaN
std,1.900278,NaN,NaN,NaN,NaN,NaN,14.233594,NaN,1.822165,1.827198,NaN
min,2021.000000,NaN,NaN,NaN,NaN,NaN,18.000000,NaN,1.000000,1.000000,NaN
25%,2021.000000,NaN,NaN,NaN,NaN,NaN,30.000000,NaN,3.000000,3.000000,NaN
50%,2021.000000,NaN,NaN,NaN,NaN,NaN,39.000000,NaN,4.000000,4.000000,NaN
75%,2025.000000,NaN,NaN,NaN,NaN,NaN,50.000000,NaN,5.000000,5.000000,NaN


In [14]:
df['colseg'].value_counts(dropna=False)
df['colseg'].value_counts(normalize=True)

colseg
Peor     0.701503
Igual    0.245409
Mejor    0.053088
Name: proportion, dtype: float64

In [15]:
for col in ['estratopri','aoj21','ur','q1','ideologia','m1','trust','pr5']:
    print(col, df[col].unique()[:20], '\n')

estratopri ['Oriental' 'Atlántica' 'Central' 'Bogotá' 'Amazonia' 'Pacífica' nan] 

aoj21 [nan 'Delincuentes comunes' 'Crimen organizado' 'Ninguno' 'Guerrillas'
 'Otros' 'Fuerzas de seguridad' 'Personas cercanas'] 

ur [nan 'Urbano' 'Rural'] 

q1 [nan 'Mujer' 'Hombre'] 

ideologia [nan 'Izquierda' 'Centro derecha' 'Derecha' 'Centro izquierda' 'Centro'] 

m1 [nan 'Malo' 'Ni bueno, ni malo (regular)' 'Muy malo (pésimo)' 'Bueno'
 'Muy bueno'] 

trust [nan  4.  7.  6.  3.  5.  2.  1.] 

pr5 [nan  3.  6.  5.  4.  2.  1.  7.] 



In [16]:
df.isna().sum()

year             0
estratopri      22
aoj21         1654
colseg        1567
ur            2994
q1            2993
q2              13
m1            1582
trust         3013
pr5           3086
ideologia     2215
dtype: int64

In [17]:
df['q2'].describe()

count    4549.000000
mean       40.792922
std        14.233594
min        18.000000
25%        30.000000
50%        39.000000
75%        50.000000
max        93.000000
Name: q2, dtype: float64

In [18]:
df.shape

(4562, 11)

In [19]:
df_model = df.copy()

# 1. Filtrar variable objetivo
df_model = df_model[df_model['colseg'].notna()]

# 2. Ordenar variable objetivo (CRÍTICO)
orden = ['Peor', 'Igual', 'Mejor']
df_model['colseg_ord'] = pd.Categorical(df_model['colseg'], categories=orden, ordered=True)

# 3. Convertir year a categórica
df_model['year'] = df_model['year'].astype(int).astype(str)

# 4. Edad limpia
df_model = df_model[df_model['q2'].notna()]

In [20]:
m1_order = ['Muy malo (pésimo)', 'Malo', 'Ni bueno, ni malo (regular)', 'Bueno', 'Muy bueno']
df_model['m1_ord'] = pd.Categorical(df_model['m1'], categories=m1_order, ordered=True).codes

In [21]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

X = df_model[['q2', 'm1_ord']]  # luego expandes con dummies
y = df_model['colseg_ord']

model = OrderedModel(y, X, distr='logit')
res = model.fit(method='bfgs')

print(res.summary())

Optimization terminated successfully.
         Current function value: 0.678093
         Iterations: 17
         Function evaluations: 22
         Gradient evaluations: 22
                             OrderedModel Results                             
Dep. Variable:             colseg_ord   Log-Likelihood:                -2022.1
Model:                   OrderedModel   AIC:                             4052.
Method:            Maximum Likelihood   BIC:                             4076.
Date:                Sun, 05 Apr 2026                                         
Time:                        21:02:07                                         
No. Observations:                2982                                         
Df Residuals:                    2978                                         
Df Model:                           2                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------

In [22]:
df_model[['year','estratopri','aoj21','q1','ur','m1','ideologia']].isna().mean()

year          0.000000
estratopri    0.002012
aoj21         0.035547
q1            0.482227
ur            0.482562
m1            0.013414
ideologia     0.221999
dtype: float64

In [26]:
X = pd.get_dummies(
    df_model[['estratopri','aoj21','ideologia','year']],
    drop_first=True
)

X['q2'] = df_model['q2']
X['m1_ord'] = df_model['m1_ord']

X = X.astype(float)

model = OrderedModel(y, X, distr='logit')
res = model.fit(method='bfgs')
print(res.summary())

Optimization terminated successfully.
         Current function value: 0.662379
         Iterations: 121
         Function evaluations: 126
         Gradient evaluations: 126
                             OrderedModel Results                             
Dep. Variable:             colseg_ord   Log-Likelihood:                -1975.2
Model:                   OrderedModel   AIC:                             3990.
Method:            Maximum Likelihood   BIC:                             4110.
Date:                Sun, 05 Apr 2026                                         
Time:                        21:06:10                                         
No. Observations:                2982                                         
Df Residuals:                    2962                                         
Df Model:                          18                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
-------------------